# COMPSCI5094 Coursework - Conversational Interfaces: Premier League Dialogue Agent
**Author**: Patricia Natasha Munginga

**Student ID**: 3033388M

**Date**: February 2026

**Project**: Task‑Oriented Dialogue System for Premier League Information

This notebook documents the development of a task‑oriented conversational agent that provides information about English Premier League teams and personnel using a football data API and an LLM‑based NLU component.
The focus is on defining intents and slots, implementing robust intent detection and slot filling with a local LLM, integrating real‑time football data via an external API and demonstrating end‑to‑end dialogues that satisfy user queries while enabling clear evaluation of NLU performance.

### Setup and Imports

In [37]:
"""Installs and Imports"""

# Install the Ollama python library
%pip install ollama

# Import the Ollama python library
import ollama

# API functionality
import requests
import json

Note: you may need to restart the kernel to use updated packages.


### Configuration

In [38]:
# Create a client connected to the local Ollama server
client = ollama.Client(host='http://localhost:11434')

# Ensure the required model is available locally (downloads it if needed)
!ollama pull qwen3:4b-instruct

# Define the model name to use throughout the notebook
MODEL = 'qwen3:4b-instruct'  

# Database and API 
FOOTBALL_DATA_API_KEY = "7b428478107f4bdb98af277f262c4350"
FOOTBALL_DATA_BASE_URL = "https://api.football-data.org/v4"
PREMIER_LEAGUE_CODE = "PL" # English Premier League

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 85e4a5b7b8ef: 100% ▕██████████████████▏ 2.5 GB                         
pulling eade0a07cac7: 100% ▕██████████████████▏ 1.4 KB                         
pulling d18a5cc71b84: 100% ▕██████████████████▏  11 KB                         
pulling 0914c7781e00: 100% ▕██████████████████▏  119 B                         
pulling b72accf9724e: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 


### API Functions

In [39]:
def call_football_data_api(endpoint, params=None, verbose=True):
    """
    Generic helper function to call the football-data API.

    endpoint: string like 'teams/57'
    params: dictionary of query parameters (e.g., {'status': 'SCHEDULED'})
    verbose: if True, prints basic debug info
    """
    headers = {'X-Auth-Token': FOOTBALL_DATA_API_KEY}
    url = f"{FOOTBALL_DATA_BASE_URL}/{endpoint}"

    try:
        response = requests.get(url, headers=headers, params=params, timeout=10)

        if verbose:
            print(f"[API] GET {url} params={params} status={response.status_code}")

        response.raise_for_status()
        return response.json()

    except requests.exceptions.RequestException as e:
        # Return a structured error (better for tool outputs)
        return {"error": str(e), "endpoint": endpoint, "params": params}

def filter_premier_league_matches(matches):
    """
    Premier League-only filter

    Purpose: /teams/{id}/matches may include PL + other competitions (e.g., CL).
    This system is Premier League only
    """
    return [m for m in matches if m.get('competition', {}).get('code') == PREMIER_LEAGUE_CODE]    
      
def get_team_id_by_name(team_name):
    """
    Helper function to get team ID from team name

    SLOT: team (required)
    Endpoint used: /v4/competitions/PL/teams
    Purpose: Resolve team name to team ID
    """
    result = call_football_data_api(f'competitions/{PREMIER_LEAGUE_CODE}/teams', verbose = False)
    
    if not result or 'error' in result:
        return None
    
    teams = result.get('teams', [])
    team_name_lower = team_name.lower().strip()
    
    # 1) Exact match on full or short name
    for team in teams:
        if team['name'].lower() == team_name_lower or \
           team['shortName'].lower() == team_name_lower:
            return team['id']
    
    # 2) Exact match on TLA (3-letter code)
    for team in teams:
        if team['tla'].lower() == team_name_lower:
            return team['id']
    
    # 3) Partial match fallback
    for team in teams:
        if team_name_lower in team['name'].lower():
            return team['id']
    
    return None 

def get_manager_info(team_name):
    """
    Get information about a team's manager/coach

    SLOT: manager
    Endpoint used: /v4/teams/{id}
    Purpose: Returns detailed team info including coach/manager field
    """
    team_id = get_team_id_by_name(team_name)

    if not team_id:
        return {"error": f"Team '{team_name}' not found in Premier League"}
    
    result = call_football_data_api(f'teams/{team_id}', verbose = False)

    if not result or 'error' in result:
        return result if result else {"error": "API call failed"}
    
    coach = result.get('coach')

    if not coach:
        return {"error": "Manager information not available"}
    
    return {
        'manager': coach.get('name', 'N/A'),
        'nationality': coach.get('nationality', 'N/A'),
        'team': result.get('name', 'N/A')
    }

def get_standings_info(team_name):
    """
    Get standings for a team

    SLOTS:
      - leaguePosition
      - numGamesPlayed
      - winLossRecord

    Endpoint used: /v4/competitions/PL/standings
    Purpose: Returns league table including position, games played, wins, draws, losses
    """

    # Step 1: Resolve team name - team ID
    team_id = get_team_id_by_name(team_name)

    if not team_id:
        return {"error": f"Team '{team_name}' not found in Premier League"}

    # Step 2: Call Premier League standings endpoint
    result = call_football_data_api(
        f'competitions/{PREMIER_LEAGUE_CODE}/standings',
        verbose = False
    )

    if not result or 'standings' not in result:
        return {"error": "Standings data not available"}

    # Step 3: Find the TOTAL standings table
    for standing_type in result['standings']:
        if standing_type['type'] == 'TOTAL':

            # Step 4: Find this team in the league table
            for team_row in standing_type['table']:
                if team_row['team']['id'] == team_id:

                    return {
                        "team": team_row['team']['name'],
                        "leaguePosition": team_row['position'],
                        "numGamesPlayed": team_row['playedGames'],
                        "winLossRecord": f"{team_row['won']}-{team_row['draw']}-{team_row['lost']}"
                    }

    return {"error": "Team not found in standings"}

def get_next_match(team_name):
    """
    Get next match for a team

    SLOTS:
       - nextGameDate
       - nextOpponent

    Endpoint used: /v4/teams/{id}/matches?status=SCHEDULED
    Purpose: Returns upcoming matches
    """

    # Step 1: Resolve team name - team ID
    team_id = get_team_id_by_name(team_name)

    if not team_id:
        return {"error": f"Team '{team_name}' not found in Premier League"}

    # Step 2: Get scheduled matches
    result = call_football_data_api(
        f'teams/{team_id}/matches',
        {'status': 'SCHEDULED'},
        verbose = False
    )

    if not result or 'matches' not in result:
        return {"error": "Match data not available"}

    matches = result['matches']

    # Step 3: Filter Premier League matches only
    pl_matches = filter_premier_league_matches(matches)

    if not pl_matches:
        return {"error": "No upcoming Premier League matches found"}

    # Step 4: Sort by utcDate ascending → next match first
    pl_matches = sorted(pl_matches, key=lambda m: m['utcDate'])
    next_match = pl_matches[0]

    # Step 5: Determine opponent
    if next_match['homeTeam']['id'] == team_id:
        opponent = next_match['awayTeam']['name']
    else:
        opponent = next_match['homeTeam']['name']

    return {
        "team": team_name,
        "nextGameDate": next_match['utcDate'],
        "nextOpponent": opponent
    }


def get_last_match(team_name):
    """
    Get last played match for a team

    SLOTS:
     - lastOpponent
     - lastScore

    Endpoint used: /v4/teams/{id}/matches?status=FINISHED
    Purpose: Returns completed matches including full-time score
    """

    # Step 1: Resolve team name → team ID
    team_id = get_team_id_by_name(team_name)

    if not team_id:
        return {"error": f"Team '{team_name}' not found in Premier League"}

    # Step 2: Get finished matches
    result = call_football_data_api(
        f"teams/{team_id}/matches",
        {"status": "FINISHED"},
        verbose = False
    )

    if not result or "matches" not in result:
        return {"error": "Match data not available"}

    matches = result["matches"]

    # Step 3: Filter Premier League matches only
    pl_matches = filter_premier_league_matches(matches)

    if not pl_matches:
        return {"error": "No finished Premier League matches found"}

    # Step 4: Sort by utcDate descending → most recent finished match first
    pl_matches = sorted(pl_matches, key=lambda m: m["utcDate"], reverse=True)
    last_match = pl_matches[0]

    # Step 5: Determine opponent
    if last_match["homeTeam"]["id"] == team_id:
        opponent = last_match["awayTeam"]["name"]
    else:
        opponent = last_match["homeTeam"]["name"]

    # Step 6: Extract score
    full_time = last_match["score"]["fullTime"]
    last_score = f"{full_time['home']} - {full_time['away']}"

    return {
        "team": team_name,
        "lastOpponent": opponent,
        "lastScore": last_score
    }


def get_playing_now(team_name):
    """
    Check if a team is currently playing a match
    
    SLOT: playingNow

    Endpoint used: /v4/teams/{id}/matches?status=IN_PLAY
    Purpose: Checks if team currently has a live match
    """

    # Step 1: Resolve team name - team ID
    team_id = get_team_id_by_name(team_name)

    if not team_id:
        return {"error": f"Team '{team_name}' not found in Premier League"}

    # Step 2: Get live matches
    result = call_football_data_api(
        f"teams/{team_id}/matches",
        {"status": "IN_PLAY"},
        verbose = False
    )

    if not result or "matches" not in result:
        return {"playingNow": False}

    matches = result["matches"]

    # Step 3: Filter Premier League matches only and Check if any PL match is live
    if not result or 'error' in result or "matches" not in result:
        return {"team": team_name, "playingNow": False}

    pl_matches = filter_premier_league_matches(result["matches"])

    return {
        "team": team_name,
        "playingNow": len(pl_matches) > 0
    }

### Tool Definitions

In [40]:
ALL_TOOLS = [

    # SLOT: manager
    {
        "type": "function",
        "function": {
            "name": "get_manager_info",
            "description": "Get the current manager of a Premier League team.",
            "parameters": {
                "type": "object",
                "properties": {
                    "team_name": {
                        "type": "string",
                        "description": "The name of the Premier League team"
                    }
                },
                "required": ["team_name"]
            }
        }
    },

    # SLOTS:
    #   - leaguePosition
    #   - numGamesPlayed
    #   - winLossRecord
    {
        "type": "function",
        "function": {
            "name": "get_standings_info",
            "description": "Get a team's current Premier League position, number of games played, and win-loss record.",
            "parameters": {
                "type": "object",
                "properties": {
                    "team_name": {
                        "type": "string",
                        "description": "The name of the Premier League team"
                    }
                },
                "required": ["team_name"]
            }
        }
    },

    # SLOTS:
    #   - nextGameDate
    #   - nextOpponent
    {
        "type": "function",
        "function": {
            "name": "get_next_match",
            "description": "Get the next Premier League match for a team including date and opponent.",
            "parameters": {
                "type": "object",
                "properties": {
                    "team_name": {
                        "type": "string",
                        "description": "The name of the Premier League team"
                    }
                },
                "required": ["team_name"]
            }
        }
    },

    # SLOTS:
    #   - lastOpponent
    #   - lastScore
    {
        "type": "function",
        "function": {
            "name": "get_last_match",
            "description": 'Get the last Premier League match for a team. Use this when the user asks who they played last, previous opponent, last game, previous match, or score of last game.',
            "parameters": {
                "type": "object",
                "properties": {
                    "team_name": {
                        "type": "string",
                        "description": "The name of the Premier League team"
                    }
                },
                "required": ["team_name"]
            }
        }
    },

    # SLOT: playingNow
    {
        "type": "function",
        "function": {
            "name": "get_playing_now",
            "description": "Check if a Premier League team is currently playing a live match.",
            "parameters": {
                "type": "object",
                "properties": {
                    "team_name": {
                        "type": "string",
                        "description": "The name of the Premier League team"
                    }
                },
                "required": ["team_name"]
            }
        }
    }

]

### System Prompt

In [ ]:
"""
SYSTEM PROMPT

Defines behavioural constraints for the LLM.

Purpose:
- Restrict assistant to Premier League domain.
- Enforce tool-grounded responses.
- Prevent hallucination and use of prior knowledge.
- Explicitly define supported slots.

This ensures the assistant behaves as a deterministic,
API-driven, task-oriented system rather than a general chatbot.
"""

SYSTEM_PROMPT = {
    "role": "system",
    "content": """
You are a task-oriented dialogue assistant for Premier League football information.

DOMAIN:
- Only Premier League teams.
- Do NOT answer questions about other leagues or competitions.

INTENT:
- The main intent is GetInfo.
- The user will ask for information about a Premier League team.

SUPPORTED SLOTS AND REQUIRED TOOL USAGE:

The user may ask for information about a Premier League team.  
When the request matches one of the following slots, you MUST call the corresponding tool.

- manager  
  If the user asks about the coach, manager, head coach, or who manages the team,  
  you MUST call get_manager_info.

- leaguePosition  
  If the user asks about league position, current standing, table position, ranking,  
  or "where are they in the league?",  
  you MUST call get_standings_info.

- numGamesPlayed  
  If the user asks how many games they have played, matches played, games so far,  
  you MUST call get_standings_info.

- winLossRecord  
  If the user asks for record, win-loss record, wins and losses, form in the league,  
  or how many wins/draws/losses they have,  
  you MUST call get_standings_info.

- nextGameDate  
  If the user asks when they play next, next match date, upcoming fixture date,  
  you MUST call get_next_match.

- nextOpponent  
  If the user asks who they play next, next opponent, upcoming opponent,  
  you MUST call get_next_match.

- lastOpponent  
  If the user asks:
- Who did they play last?
- Who was their previous opponent?
- Who was their last opponent?
- What was their last game?
- What happened in their previous match?
You MUST call get_last_match.

- lastScore  
  If the user asks what was the score of the last game, result of the previous match,  
  or how the last match ended,  
  you MUST call get_last_match.

- playingNow  
  If the user asks whether the team is playing right now, currently playing,  
  in a live match, current match status, or similar wording,  
  you MUST call get_playing_now.

CRITICAL RULES:
1. You MUST call the appropriate tool to retrieve information.
2. NEVER use prior knowledge.
3. ONLY use information returned by tools.
4. If a tool returns an error, politely inform the user.
5. If a request is outside the Premier League domain, politely decline.
6. Keep responses clear, natural, and conversational.
7. If the user refers to a team using pronouns like "they" or "their", assume they are referring to the most recently mentioned team.

You are a factual football information assistant, not a general chatbot.
"""
}

### Dialogue Manager

In [42]:
"""
DIALOGUE MANAGER

Responsible for:
1. Receiving tool calls from the LLM (intent + slot extraction).
2. Executing the corresponding backend function.
3. Returning structured API results back to the LLM.
4. Generating the final natural language response.

This separates:
- NLU (model selects tool)
- Backend execution (Python functions + API)
- NLG (model generates final response)

Ensures deterministic, tool-grounded behaviour.
"""
def execute_tool(tool_call):
    """
    Execute the appropriate backend function based on the
    LLM-selected tool name.
    """

    function_name = tool_call.function['name']
    arguments = tool_call.function['arguments']

    if function_name == 'get_manager_info':
        return get_manager_info(arguments['team_name'])

    elif function_name == 'get_standings_info':
        return get_standings_info(arguments['team_name'])

    elif function_name == 'get_next_match':
        return get_next_match(arguments['team_name'])

    elif function_name == 'get_last_match':
        return get_last_match(arguments['team_name'])

    elif function_name == 'get_playing_now':
        return get_playing_now(arguments['team_name'])

    else:
        return {"error": f"Unknown function: {function_name}"}


### Main Loop

In [43]:
def run_football_agent():
    """
    Main conversation loop

    Prints dialogue in coursework annotated format:
    USER:
    - Intent:
    - Slots:

    Ends automatically when intent = Other
    """

    messages = [SYSTEM_PROMPT]

    print("ASSISTANT: Hello, how can I help you?\n")
    messages.append({"role": "assistant", "content": "Hello, how can I help you?"})

    MAX_TURNS = 20
    turn_count = 0

    # Dialogue State Memory
    # Stores last mentioned team to support pronouns like "they"
    last_team = None

    while turn_count < MAX_TURNS:

        user_message = input("USER: ").strip()

        if not user_message:
            continue

        print(f"USER: {user_message}\n")
        messages.append({"role": "user", "content": user_message})

        # NLU Step
        response = client.chat(
            model=MODEL,
            messages=messages,
            tools=ALL_TOOLS,
            options={"temperature": 0.2}
        )

        # Default intent
        detected_intent = "Other"
        detected_slots = {}

        tool_call = None

        if response.message.tool_calls:

            tool_call = response.message.tool_calls[0]
            function_name = tool_call.function["name"]
            arguments = tool_call.function["arguments"]

            detected_intent = "GetInfo"

            # Extract team if present
            team_name = arguments.get("team_name")

            if team_name:
                last_team = team_name  # Update memory
            else:
                team_name = last_team  # Use memory

            # Map function to info slot
            info_mapping = {
                "get_manager_info": "manager",
                "get_standings_info": "winLossRecord",
                "get_next_match": "nextOpponent",
                "get_last_match": "lastScore",
                "get_playing_now": "playingNow"
            }

            detected_slots = {
                "team": team_name,
                "info": info_mapping.get(function_name, "unknown")
            }

            # Inject remembered team back into tool arguments if needed
            if not arguments.get("team_name") and last_team:
                tool_call.function["arguments"]["team_name"] = last_team

        # Print Annotated USER Turn
        print(f"- Intent: {detected_intent}")
        print(f"- Slots: {detected_slots}\n")

        # If intent = Other to Exit
        if detected_intent == "Other":
            print("ASSISTANT: Bye.\n")
            break

        # Execute Tool
        result = execute_tool(tool_call)

        messages.append({
            "role": "tool",
            "content": json.dumps(result)
        })

        # NLG Step
        final_response = client.chat(
            model=MODEL,
            messages=messages,
            options={"temperature": 0.6}
        )

        assistant_message = final_response.message.content

        print(f"ASSISTANT: {assistant_message}\n")

        messages.append({"role": "assistant", "content": assistant_message})

        turn_count += 1

    return messages



if __name__ == "__main__":
    run_football_agent()

ASSISTANT: Hello, how can I help you?

USER: Hello, I need to know the record of Manchester City?

- Intent: GetInfo
- Slots: {'team': 'Manchester City', 'info': 'winLossRecord'}

ASSISTANT: Manchester City's win-loss record in the Premier League is 16 wins, 5 losses, and 5 draws.

USER: Are they playing right now?

- Intent: GetInfo
- Slots: {'team': 'Manchester City FC', 'info': 'playingNow'}

ASSISTANT: No, Manchester City is not currently playing a match.

USER: Who did they play last?

- Intent: Other
- Slots: {}

ASSISTANT: Bye.

